#  03 : Develop a Sentiment Analysis Model to Analyze Customer Reviews using Amazon Product Review / IMDB Dataset and Evaluate its Performance.

# Step 1: Import Libraries

In [10]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 2: Load Dataset

In [11]:
data = pd.read_csv("amazon_reviews.csv", nrows=50000)

print(data.head())
print(data.shape)

    AKM1MP6P0OYPR  0132793040  5.0  1365811200
0  A2CX7LUOHB2NDG  0321732944  5.0  1341100800
1  A2NWSAGRHCP8N5  0439886341  1.0  1367193600
2  A2WNBOD3WNDNKT  0439886341  3.0  1374451200
3  A1GI0U4ZRJA8WN  0439886341  1.0  1334707200
4  A1QGNMC6O1VW39  0511189877  5.0  1397433600
(50000, 4)


# Step 3: Select Required Columns

In [14]:
print(data.columns)

Index(['AKM1MP6P0OYPR', '0132793040', '5.0', '1365811200'], dtype='object')


In [16]:
data = pd.read_csv(
    "amazon_reviews.csv",
    nrows=50000,
    header=None,
    names=['reviewerID', 'asin', 'overall', 'unixReviewTime']
)

print(data.head())
print(data.columns)

       reviewerID        asin  overall  unixReviewTime
0   AKM1MP6P0OYPR  0132793040      5.0      1365811200
1  A2CX7LUOHB2NDG  0321732944      5.0      1341100800
2  A2NWSAGRHCP8N5  0439886341      1.0      1367193600
3  A2WNBOD3WNDNKT  0439886341      3.0      1374451200
4  A1GI0U4ZRJA8WN  0439886341      1.0      1334707200
Index(['reviewerID', 'asin', 'overall', 'unixReviewTime'], dtype='object')


In [17]:
data = data[['overall']]

In [18]:
data['sentiment'] = data['overall'].apply(lambda x: 1 if x >= 3 else 0)

In [19]:
data['reviewText'] = data['overall'].astype(str)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    data['reviewText'],
    data['sentiment'],
    test_size=0.2,
    random_state=42
)

# Step 4: Convert Rating to Sentiment

In [21]:
data['sentiment'] = data['overall'].apply(lambda x: 1 if x >= 3 else 0)

print(data.head())

   overall  sentiment reviewText
0      5.0          1        5.0
1      5.0          1        5.0
2      1.0          0        1.0
3      3.0          1        3.0
4      1.0          0        1.0


# Step 5: Create Text Data

In [22]:
data['reviewText'] = data['overall'].astype(str)

# Step 6: Split Dataset

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data['reviewText'],
    data['sentiment'],
    test_size=0.2,
    random_state=42
)

# Step 7: Tokenization

In [24]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Step 8: Padding

In [25]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

# Step 8: Build Model

In [26]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(5000, 64, input_length=100),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

C:\Users\kambl\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# Step 9: Compile Model

In [27]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Step 10: Train Model

In [28]:
history = model.fit(X_train_pad, y_train, epochs=5, validation_data=(X_test_pad, y_test))

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - accuracy: 0.9591 - loss: 0.0968 - val_accuracy: 1.0000 - val_loss: 8.5162e-05
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 41s 33ms/step - accuracy: 1.0000 - loss: 6.1170e-05 - val_accuracy: 1.0000 - val_loss: 1.9714e-05
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 61s 49ms/step - accuracy: 1.0000 - loss: 1.6313e-05 - val_accuracy: 1.0000 - val_loss: 8.4477e-06
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 50s 40ms/step - accuracy: 1.0000 - loss: 7.1288e-06 - val_accuracy: 1.0000 - val_loss: 4.0666e-06
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 62s 50ms/step - accuracy: 1.0000 - loss: 3.5363e-06 - val_accuracy: 1.0000 - val_loss: 2.0643e-06


# Step 11: Evaluate Model

In [30]:
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 1.0000 - loss: 2.0451e-06
Accuracy: 1.0
